# Project 2: Services Architecture - Initial Investigation

## Purpose

We all learn by reading and experimenting.  To get into that mode, we will
review some documentation and run some pre-existing tests. Based on the 
test results, we will make some preliminary observations on the behaviors
of interfaces and start developing ideas on how we can leverage them in
the architecture we will be developing.

## Investigation

There are three main services available that are to be used to build the 
application. These services do not need to be changed. Your application will 
build on top of them. The services are:

- A [Document Repository](http://seappserver1.rit.edu/DMService): This is where the incoming applications (scanned images of submitted applications) are stored.
- An [OCR Service](http://seappserver1.rit.edu/OCRService): This service takes documents (scanned images) and converts the scanned information to text
- A [Parsing Service](http://seappserver1.rit.edu/ParserService): This service takes the textual information and extracts the key data for tracking purposes. 

We will start by more closely examining each of these services.  The main overview for [Forms Services](https://seappserver1.rit.edu/formsservices/) may also be useful.  

## Document Repository

The **Document Repository** as its name implies is where scanned image documents
reside.  We will need to be able to list the contents of the repository and 
retrieve files for processing. 

### List Files
The `List Files` test in our test application shows this being done within a C#
program.  (Run this before proceeding further)

In [1]:
# Run 1 - GetFileList


The output shows us a couple critical pieces of information that will be useful
as we go forward.  First off, it tells us the number of files in the repository.

We have constructed this as a testbed and divorced the document repository from
the document scanning function.  This means the number of documents will remain
constant throughout this project.  However, we as architects realize that our
while our test environment shows a constant number of documents, our production
system will differ and contantly have new documents arriving.

There are two important implications - the number of document will grow over 
time and our system must somehow now which documents have been processed and
ensure that we never resubmit the same document twice.

The documentation for the `ListFiles` call (avaialable through the *Document Repository*
link above) shows us it is a RESTful API call.  Our C# code provided the 
framework needed to make the call and demonstrated that output can be returned
in two different formats - XML or JSON.  

#### Practice: Output Formats

The following code block provides the basic instrumentation to call `ListFiles`,
you need to complete the call and return XML, and also Json

In [2]:
import requests

def get_dm_files():
    url = "http://seappserver1.rit.edu/dmservice/api/listfiles"
    hdr_xml = {"Accept": "application/xml"}
    response_xml = requests.get(url, headers=hdr_xml)
    print("=== XML Response ===")
    print(response_xml.text)
    print()
    hdr_json = {"Accept": "application/json"}
    response_json = requests.get(url, headers=hdr_json)
    print("=== JSON Response ===")
    print(response_json.text)
    print()
    return response_json.json()

file_list = get_dm_files()
print(f"Total files found: {len(file_list)}")
print(f"First entry: {file_list[0]}")


=== XML Response ===
<ArrayOfListFilesResult xmlns:i="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://schemas.datacontract.org/2004/07/DMService.Controllers"><ListFilesResult><fileName>Application_L_Page_002.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_003.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_004.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_005.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_006.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_007.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_008.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_009.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_010.png</fileName></ListFilesResult><ListFilesResult><fileName>Application_L_Page_011.png</fileName></ListFilesResult><ListFilesRes

#### Practice: File Retrieval

Complete the code below to retrieve a single file.  You should be able to pick any file name from the prior list of files 

In [3]:
import requests

def get_one_dm_file(filename=None):
    print("Getting file")
    if filename is None:
        list_url  = "http://seappserver1.rit.edu/dmservice/api/listfiles"
        file_list = requests.get(list_url, headers={"Accept": "application/json"}).json()
        filename = file_list[0]['fileName']
        print(f"Auto-selected file: {filename}")
    url = f"http://seappserver1.rit.edu/dmservice/api/downloadfile?filename={filename}"
    response = requests.get(url)
    response.raise_for_status()
    local_path = "./dm_file.png"
    with open(local_path, "wb") as f:
        f.write(response.content)
    print(f"Received file: {filename}; Saved as: {local_path} ({len(response.content)} bytes)")
    return local_path

downloaded_file = get_one_dm_file()


Getting file
Auto-selected file: Application_L_Page_002.png
Received file: Application_L_Page_002.png; Saved as: ./dm_file.png (152022 bytes)


#### Practice: File Processing

Once you have an image file (.png, .jpg ...), you need to extract the data.  The OCR service provides several ways to do this.  Look at the testing app provided to experiment offline, and we will have you replicate the behaviour in this workbook.
Write python code to pick take the file you retrieved from the `getfile` API, and then use that file to submit for processing using using the `processfile` API.  Save the text output from the service in a local file and print the contents to the console.
Starter code is provided.  Fill in the rest.  


In [4]:
import requests

def post_ocr_process_file(local_image_path="./dm_file.png"):
    url = "http://seappserver1.rit.edu/ocrservice/api/processfile"
    with open(local_image_path, 'rb') as f:
        result = requests.post(
            url,
            files={'file': ('dm_file.png', f, 'image/png')}
        )
    result.raise_for_status()
    ocr_data = result.json()
    print("=== OCR Result (JSON) ===")
    print(ocr_data)
    raw_text = ocr_data.get('_text', str(ocr_data))
    output_path = "./ocr_result.txt"
    with open(output_path, 'w') as out:
        out.write(raw_text)
    print(f"\nText saved to: {output_path}")
    print("\n=== Extracted Text ===")
    print(raw_text)
    return raw_text

ocr_text = post_ocr_process_file(downloaded_file)


=== OCR Result (JSON) ===
{'_text': 'Registration Application\n\nFirst Name: Fidel\n\nLast Name: Beer\n\nApplication Type: new\n\nAddress: 433 Runolfsson Corner\n\nCity: West Reina\n\nDate: 7/24/1983 12:00:00 AM\n\nEmail: Enola_Williamson@elfrieda.ca\nPhone: 669.748.2100\n\nDescription: voluptate ea a eos et qui qui nam iste voluptate ea a eos et qui qui nam iste voluptate ea a eos\net qui qui nam istevoluptate ea a eos et qui qui nam istevoluptate ea a eos et qui qui nam istevoluptate ea a\neos et qui qui nam istevoluptate ea a eos et qui qui nam istevoluptate ea a eos et qui qui nam istevoluptate\nea a eos et qui qui nam iste voluptate ea a eos et qui qui nam iste voluptate ea a eos et qui qui nam\nistevoluptate ea a eos et qui qui nam istevoluptate ea a eos et qui qui nam istevoluptate ea a eos et qui qui\nnam istevoluptate ea a eos et qui qui nam istevoluptate ea a eos et qui qui nam istevoluptate ea a eos et qui\nqui nam iste voluptate ea a eos et qui qui nam iste voluptate ea a e

#### Practice: File Processing - Asynchronous
Not all work done by software services is instantaneous.  In fact, many modern services take seconds to multiple minutes to 'do their job'.  In the prior example, we took the lazy programming approach and just make the user wait till the job is done.  This works, but it prevents the user from being able to do anything else.  And what if the processing work took many minutes, or even hours?  Today, vision analytics and algorithms that take in big-data for machine learning can easily take that much time.  To allow parallel processing, is it necessary to provide APIs that enable asynchronous behaviour. i.e. Submit something for processing but don't wait for it.  Just come back from time to time and see if the job is completed.  
We will have to experiment with this time of behaviour.
You will use the `processfileasync` API, but will also need to implement a mechanism to check to see *when* the work is completed.


In [5]:
import requests
import time

def monitor(filePath):
    count = 0
    url = "http://seappserver1.rit.edu/OCRService/api/GetProcessedFile?filename="
    outputFile = ""
    while count < 10:
        response = requests.get(url + filePath)
        print(str(count) + ":" + str(response))
        print(response.json())
        count = count + 1
        if response.json()['_fileReady'] == False:
            print(f"[{count}]: File {filePath} is not ready")
            time.sleep(5)
        else:
            outputFile = response.json()['_fileData']
            break
    return outputFile

def ocr_async(local_image_path="./dm_file.png"):
    url = "http://seappserver1.rit.edu/ocrservice/api/processfileasync"
    outputFile = ""
    with open(local_image_path, 'rb') as f:
        result = requests.post(
            url,
            files={'file': ('dm_file.png', f, 'image/png')}
        )
    result.raise_for_status()
    print("Async submission response:", result.json())
    response_data = result.json()
    outputFile = response_data.get('_outputFilePath', '')
    print("Monitor for: " + outputFile)
    ocrData = monitor(outputFile)
    if ocrData == "":
        print("OCR Failed - timed out after 10 attempts")
    else:
        destFile = 'async_ocr_result.txt'
        print("OCR Result stored in " + destFile)
        with open(destFile, 'w') as f:
            f.write(ocrData)
        print("=== Async OCR Result ===")
        print(ocrData)
    return ocrData

async_ocr_text = ocr_async("./dm_file.png")


Async submission response: {'_inputFile': 'dm_file.png', '_inputFileLength': 152022, '_outputFilePath': 'b68381dc-2823-4b95-bf0b-3fc534c75f97/dm_file.txt'}
Monitor for: b68381dc-2823-4b95-bf0b-3fc534c75f97/dm_file.txt
0:<Response [200]>
{'_fileData': '', '_fileReady': False, '_fileName': 'b68381dc-2823-4b95-bf0b-3fc534c75f97/dm_file.txt'}
[1]: File b68381dc-2823-4b95-bf0b-3fc534c75f97/dm_file.txt is not ready
1:<Response [200]>
{'_fileData': '', '_fileReady': False, '_fileName': 'b68381dc-2823-4b95-bf0b-3fc534c75f97/dm_file.txt'}
[2]: File b68381dc-2823-4b95-bf0b-3fc534c75f97/dm_file.txt is not ready
2:<Response [200]>
{'_fileData': '', '_fileReady': False, '_fileName': 'b68381dc-2823-4b95-bf0b-3fc534c75f97/dm_file.txt'}
[3]: File b68381dc-2823-4b95-bf0b-3fc534c75f97/dm_file.txt is not ready
3:<Response [200]>
{'_fileData': 'Registration Application\n\nFirst Name: Fidel\n\nLast Name: Beer\n\nApplication Type: new\n\nAddress: 433 Runolfsson Corner\n\nCity: West Reina\n\nDate: 7/24/1983 

#### Putting it all together
So, we have experimented with two services
1. A document repository (DMService) that holds a set of files
2. An OCR service that converts images to text (and we did it two different ways)

At the end of this, we have text data from pictures.  Great.  And we would want to use that data.  Not just store blobs of text, but actually put that into a more organized form.  
- How about a database?  If we look at the text result, it is a set of fields, and there is data for each field.  Sounds a lot like a database table.

We have one more service to help us take plain text, and make it more 'organized' i.e. `field: value` as a nice data structure, which will allow us to easily store into a DB (we'll leave out the actual DB for now!)

In this final exercise, use the additional API for Parser Service for ReadForm.  
You can read more about the API on the website.
You will now put together all the pieces ... which is how you build pretty much all applications through integration of multiple distributed components  

1. Retrieve a file using `GetFile`
2. Process a file using one of the `ProcessFile...` APIs
3. Convert the raw text into a json (name:value) list using the `ReadForm` API and save that json formatted file
    - As mentioned above, you would then store the data into a table in a read DB, but we'll leave that for your own experimentation!  


Some starter python code is provided below.  Fill in the rest using what you learned in the prior steps ... and make the integrated application work!  

In [6]:
import requests
import json

LOCAL_IMAGE = "./dm_file.png"
LOCAL_OCR   = "./ocr_result.txt"
LOCAL_JSON  = "./parsed_form.json"

def get_a_file():
    print("[1] Getting a file from the Document Repository...")
    list_url  = "http://seappserver1.rit.edu/dmservice/api/listfiles"
    file_list = requests.get(list_url, headers={"Accept": "application/json"}).json()
    filename  = file_list[0]['fileName']
    download_url = f"http://seappserver1.rit.edu/dmservice/api/downloadfile?filename={filename}"
    response = requests.get(download_url)
    response.raise_for_status()
    with open(LOCAL_IMAGE, "wb") as f:
        f.write(response.content)
    print(f"    Downloaded '{filename}' → {LOCAL_IMAGE} ({len(response.content)} bytes)")

def process_a_file():
    print("[2] Processing file through OCR service...")
    ocr_url = "http://seappserver1.rit.edu/ocrservice/api/processfile"
    with open(LOCAL_IMAGE, 'rb') as f:
        result = requests.post(
            ocr_url,
            files={'file': ('dm_file.png', f, 'image/png')}
        )
    result.raise_for_status()
    raw_text = result.json().get('_text', str(result.json()))
    with open(LOCAL_OCR, 'w') as out:
        out.write(raw_text)
    print(f"    OCR complete → {LOCAL_OCR}")

def parse_a_file():
    print("[3] Parsing OCR text into structured JSON via Parser service...")
    with open(LOCAL_OCR, 'r') as f:
        raw_text = f.read()
    parser_url = "http://seappserver1.rit.edu/parserservice/api/readform"
    response = requests.post(
        parser_url,
        files={"file": ("ocr.txt", raw_text)},
        headers={"Accept": "application/json"}
    )
    response.raise_for_status()
    parsed_data = response.json()
    with open(LOCAL_JSON, 'w') as out:
        json.dump(parsed_data, out, indent=4)
    print(f"    Parsing complete → {LOCAL_JSON}")
    return parsed_data

print("Running the integrated application")
print("=" * 50)
get_a_file()
process_a_file()
final_json = parse_a_file()
print()
print("=" * 50)
print("Final structured form data (JSON):")
print(json.dumps(final_json, indent=4))


Running the integrated application
[1] Getting a file from the Document Repository...
    Downloaded 'Application_L_Page_002.png' → ./dm_file.png (152022 bytes)
[2] Processing file through OCR service...
    OCR complete → ./ocr_result.txt
[3] Parsing OCR text into structured JSON via Parser service...
    Parsing complete → ./parsed_form.json

Final structured form data (JSON):
{
    "allFields": [
        {
            "fieldName": "First Name",
            "fieldValue": " Fidel"
        },
        {
            "fieldName": "Last Name",
            "fieldValue": " Beer"
        },
        {
            "fieldName": "Application Type",
            "fieldValue": " new"
        },
        {
            "fieldName": "Address",
            "fieldValue": " 433 Runolfsson Corner"
        },
        {
            "fieldName": "City",
            "fieldValue": " West Reina"
        },
        {
            "fieldName": "Email",
            "fieldValue": " Enola_Williamson@elfrieda.ca"
      

#### Conclusions

Now that you have a working application, run the application we provided (the C# app) and run ALL the commands.  Observe the output, watch the behaviour.  Think about how the operations work (and why).  Consider the choices to be made in putting the service components together.  Add your thoughts below

#### Student observations






